# Presentation Visuals

Produces DAIR-branded maps of registered active pharmacies in Gauteng and KwaZulu-Natal, overlaid on 2020 MDB ward boundaries.

The notebook first performs a spatial QA pass — joining pharmacy points to ward polygons, checking for province mismatches, and exporting suspicious records. The final maps use a consistent design language with DAIR brand colors.

Inputs: `PHARMACIES_MASTER_FINAL.csv`, `SA_Wards2020.shp`

Outputs: `DAIR_pharmacy_map.png`, `pharmacies_outside_wards.csv`, `pharmacies_province_mismatches.csv`

## Imports and configuration

In [ ]:
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.ticker import NullFormatter

DAIR = {
    'off_white':     '#FCFFEB',
    'paper_beige':   '#EFF1E3',
    'carton_beige':  '#F9EECE',
    'vermillion':    '#C8441A',
    'yellow':        '#FFCB05',
    'verdant_green': '#007264',
    'navy':          '#2E3E6C',
    'brown':         '#442520',
    'purple':        '#7D64AD',
}

BOX       = r'C:\Users\jcahi\Box\DAIR_SA\Data Sets\Pharmacy Data\Cleaned Datasets'
WARDS_DIR = r'C:\Users\jcahi\Box\DAIR_SA\Data Sets\Spatial Data\MDB_Wards_2020_-7628782095861510160'

## Load data

In [ ]:
df = pd.read_csv(f'{BOX}\\PHARMACIES_MASTER_FINAL.csv', dtype=str)
df['LAT'] = pd.to_numeric(df['LAT'], errors='coerce')
df['LNG'] = pd.to_numeric(df['LNG'], errors='coerce')

df_geo  = df[df['LAT'].notna() & df['LNG'].notna()].copy()
gauteng = df_geo[df_geo['PROVINCE'].str.lower().str.contains('gauteng', na=False)].copy()
kzn     = df_geo[df_geo['PROVINCE'].str.lower().str.contains('kwazulu|natal', na=False)].copy()

print(f'Geocoded pharmacies:')
print(f'  Gauteng : {len(gauteng):,}')
print(f'  KZN     : {len(kzn):,}')

wards = gpd.read_file(WARDS_DIR)
wards = wards.to_crs('EPSG:4326')

# Repair geometries
wards = wards[wards.geometry.notna() & ~wards.geometry.is_empty].copy()
wards['geometry'] = wards.geometry.make_valid()
wards = wards.explode(index_parts=False).reset_index(drop=True)
wards = wards[wards.geometry.type.isin(['Polygon', 'MultiPolygon'])].copy()

print(f'Invalid ward geometries after repair: {(~wards.is_valid).sum()}')

# Detect province column
prov_col = next(
    (c for c in wards.columns
     if c.upper() in ('PROVINCE', 'PROV_NAME', 'PROVINCE_N', 'PROVINCE_NAME', 'PR_NAME', 'PNAME')),
    None
)
print(f'Province column: {prov_col}')
if prov_col:
    print('Unique values:', wards[prov_col].unique())

## Subset wards and build province outlines

In [ ]:
if prov_col:
    wards_gp  = wards[wards[prov_col].astype(str).str.contains('Gauteng', case=False, na=False)].copy()
    wards_kzn = wards[wards[prov_col].astype(str).str.contains('KwaZulu', case=False, na=False)].copy()
else:
    # Fallback to bounding box if no province column found
    wards_gp  = wards.cx[27.4:29.2, -26.8:-25.1].copy()
    wards_kzn = wards.cx[28.7:32.9, -31.2:-26.7].copy()

print(f'Gauteng wards : {len(wards_gp):,}')
print(f'KZN wards     : {len(wards_kzn):,}')

prov_gp  = gpd.GeoDataFrame(geometry=[wards_gp.geometry.union_all().buffer(0)], crs=wards_gp.crs)
prov_kzn = gpd.GeoDataFrame(geometry=[wards_kzn.geometry.union_all().buffer(0)], crs=wards_kzn.crs)

gauteng_gdf = gpd.GeoDataFrame(gauteng, geometry=gpd.points_from_xy(gauteng['LNG'], gauteng['LAT']), crs='EPSG:4326')
kzn_gdf     = gpd.GeoDataFrame(kzn,     geometry=gpd.points_from_xy(kzn['LNG'],     kzn['LAT']),     crs='EPSG:4326')

## Spatial QA

Joins pharmacy points to ward polygons and flags province mismatches and unmatched points. Exports both to CSV for review.

In [ ]:
def norm_province(x):
    if pd.isna(x): return None
    return ' '.join(str(x).strip().lower().replace('-', ' ').split())

# Spatial join — assigns each pharmacy to a ward
wards_for_join = wards[[prov_col, 'wardid', 'wardlabel', 'geometry']].copy() if prov_col else wards[['geometry']].copy()
gauteng_joined = gpd.sjoin(gauteng_gdf, wards_for_join, how='left', predicate='within')

unmatched = gauteng_joined[gauteng_joined['wardid'].isna()].copy()
print(f'Gauteng pharmacies outside all ward polygons: {len(unmatched)}')

# Province mismatch check
if prov_col and 'province_right' in gauteng_joined.columns:
    gauteng_joined['input_province_norm'] = gauteng_joined['PROVINCE'].map(norm_province)
    gauteng_joined['ward_province_norm']  = gauteng_joined['province_right'].map(norm_province)
    mismatches = gauteng_joined[
        gauteng_joined['wardid'].notna() &
        (gauteng_joined['input_province_norm'] != gauteng_joined['ward_province_norm'])
    ].copy()
    print(f'Province mismatches: {len(mismatches)}')
else:
    mismatches = pd.DataFrame()

# Export suspicious records
suspect_cols = [c for c in ['pharmacy_id', 'PRACTICE_NAME', 'PROVINCE', 'matched_address', 'LAT', 'LNG', 'api_status']
                if c in gauteng_joined.columns]

unmatched[suspect_cols].to_csv(f'{BOX}\\pharmacies_outside_wards.csv', index=False)
if len(mismatches):
    mismatches[suspect_cols].to_csv(f'{BOX}\\pharmacies_province_mismatches.csv', index=False)
print('QA files exported.')

## Spatial filter — keep only pharmacies inside province boundaries

In [ ]:
def filter_inside(gdf, prov_gdf):
    inside = gpd.sjoin(gdf, prov_gdf[['geometry']], predicate='within', how='inner')
    return inside.drop(columns='index_right')

gauteng_inside = filter_inside(gauteng_gdf, prov_gp)
kzn_inside     = filter_inside(kzn_gdf,     prov_kzn)

print('Pharmacies outside province boundary:')
print(f'  Gauteng : {len(gauteng_gdf) - len(gauteng_inside):,}')
print(f'  KZN     : {len(kzn_gdf) - len(kzn_inside):,}')
print()
print('Pharmacies retained for mapping:')
print(f'  Gauteng : {len(gauteng_inside):,}')
print(f'  KZN     : {len(kzn_inside):,}')

# Save excluded points for QA
gauteng_gdf[~gauteng_gdf.index.isin(gauteng_inside.index)].to_csv(f'{BOX}\\gauteng_pharmacies_outside_boundary.csv', index=False)
kzn_gdf[~kzn_gdf.index.isin(kzn_inside.index)].to_csv(f'{BOX}\\kzn_pharmacies_outside_boundary.csv', index=False)

## Map helper function

In [ ]:
plt.rcParams.update({
    'figure.dpi': 140,
    'font.size': 10,
    'axes.titlesize': 14,
    'axes.titleweight': 'bold',
})


def make_map(ax, pharmacies, wards_gdf, prov_gdf, title, dot_color, bounds):
    lng_min, lng_max, lat_min, lat_max = bounds

    ax.set_facecolor(DAIR['carton_beige'])

    wards_gdf.plot(ax=ax, color=DAIR['paper_beige'], edgecolor='#d0c8b8', linewidth=0.18, zorder=1)
    prov_gdf.boundary.plot(ax=ax, color=DAIR['navy'], linewidth=1.2, zorder=2)

    ax.scatter(pharmacies['LNG'], pharmacies['LAT'],
               color=dot_color, s=7, alpha=0.75, linewidths=0, zorder=3)

    ax.set_xlim(lng_min, lng_max)
    ax.set_ylim(lat_min, lat_max)
    ax.xaxis.set_major_formatter(NullFormatter())
    ax.yaxis.set_major_formatter(NullFormatter())
    ax.tick_params(left=False, bottom=False)

    for spine in ax.spines.values():
        spine.set_edgecolor('#aaaaaa')
        spine.set_linewidth(0.5)

    ax.set_title(title, fontsize=11, fontweight='bold', fontfamily='monospace',
                 color=DAIR['navy'], pad=10, loc='left')

    ax.annotate(f'n = {len(pharmacies):,} pharmacies',
                xy=(0.98, 0.04), xycoords='axes fraction',
                fontsize=7.5, color='#666666', ha='right', fontfamily='monospace')

    # Scale bar
    centre_lat  = (lat_min + lat_max) / 2
    km_per_deg  = 111.32 * np.cos(np.radians(centre_lat))
    map_width_km = (lng_max - lng_min) * km_per_deg
    raw         = map_width_km / 5
    scale_km    = round(raw / 10) * 10 if raw > 10 else round(raw / 5) * 5
    scale_deg   = scale_km / km_per_deg
    bar_x = lng_min + (lng_max - lng_min) * 0.05
    bar_y = lat_min + (lat_max - lat_min) * 0.05
    bar_h = (lat_max - lat_min) * 0.008

    ax.fill_between([bar_x, bar_x + scale_deg], bar_y, bar_y + bar_h,
                    color=DAIR['navy'], zorder=5)
    ax.annotate(f'{scale_km} km',
                xy=(bar_x + scale_deg / 2, bar_y + bar_h * 2.5),
                fontsize=7, color=DAIR['navy'], ha='center',
                fontfamily='monospace', zorder=5)

    ax.legend(handles=[mpatches.Patch(color=dot_color, label='Pharmacy')],
              loc='upper right', fontsize=7, frameon=True, framealpha=0.85,
              edgecolor='#cccccc', handlelength=1)

## Draw and export maps

In [ ]:
GAUTENG_BOUNDS = (27.15, 29.25, -26.85, -25.00)
KZN_BOUNDS     = (28.80, 32.90, -31.15, -26.80)

fig = plt.figure(figsize=(16, 8), facecolor=DAIR['off_white'])
gs  = fig.add_gridspec(1, 2, width_ratios=[1, 1], wspace=0.04,
                        left=0.03, right=0.97, top=0.88, bottom=0.08)

ax1 = fig.add_subplot(gs[0])
ax2 = fig.add_subplot(gs[1])
ax1.set_box_aspect(1)
ax2.set_box_aspect(1)

make_map(ax1, gauteng_inside, wards_gp,  prov_gp,  'Gauteng',       DAIR['vermillion'],    GAUTENG_BOUNDS)
make_map(ax2, kzn_inside,     wards_kzn, prov_kzn, 'KwaZulu-Natal', DAIR['verdant_green'], KZN_BOUNDS)

fig.suptitle('Registered Active Pharmacies',
             fontsize=14, fontweight='bold', fontfamily='monospace',
             color=DAIR['navy'], y=0.97, x=0.03, ha='left')

fig.text(0.03, 0.92,
         'Gauteng and KwaZulu-Natal  ·  Source: SAPC / Google Places  ·  Ward boundaries: MDB 2020',
         fontsize=7.5, color='#888888', fontfamily='monospace', ha='left')

fig.text(0.50, 0.025,
         'Maps shown in equal-sized panels; scale differs by province.',
         fontsize=7.5, color='#999999', fontfamily='monospace', ha='center')

OUTPUT = f'{BOX}\\DAIR_pharmacy_map.png'
plt.savefig(OUTPUT, dpi=300, bbox_inches='tight', facecolor=DAIR['off_white'])
plt.show()
print(f'Saved -> {OUTPUT}')